In [2]:
# Cell 2: Core imports

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from google import genai


In [ ]:
import os
from google import genai
import time

os.environ["GEMINI_API_KEY"] = "-_-"

GEMINI_MODEL = "text-embedding-004"
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def embed_texts_with_gemini(texts, batch_size=16, max_retries=3, sleep_base=2.0):
    """
    texts: list[str]
    returns: np.ndarray of shape (n_samples, embedding_dim)

    - Smaller batch_size to reduce load
    - Retries on ServerError with exponential backoff
    """
    all_vecs = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]

        # retry loop for this batch
        last_exc = None
        for attempt in range(max_retries):
            try:
                resp = gemini_client.models.embed_content(
                    model=GEMINI_MODEL,
                    contents=batch,
                )
                for emb in resp.embeddings:
                    all_vecs.append(emb.values)
                break  # success → break retry loop
            except ServerError as e:
                last_exc = e
                # exponential backoff before retry
                wait = sleep_base * (attempt + 1)
                print(f"[embed_texts_with_gemini] ServerError on batch {start}:{start+len(batch)}, "
                      f"attempt {attempt+1}/{max_retries} → retrying after {wait:.1f}s")
                time.sleep(wait)
            except APIError as e:
                # For client-side errors (auth/quota/etc.), no point retrying
                print("[embed_texts_with_gemini] APIError (non-retryable):", e)
                raise

        else:
            # If we exhausted retries for this batch
            print("[embed_texts_with_gemini] FAILED after retries on batch "
                  f"{start}:{start+len(batch)}; raising last exception")
            raise last_exc

    return np.array(all_vecs, dtype=np.float32)


In [7]:
import pandas as pd

# Cell 4 (fixed): Load and merge GoEmotions splits as proper CSV

files = [
    "emotion_data/data/full_dataset/goemotions_1.csv",  # TODO: change to your actual filenames
    "emotion_data/data/full_dataset/goemotions_2.csv",
    "emotion_data/data/full_dataset/goemotions_3.csv",
]

dfs = []
for path in files:
    # try standard CSV parsing first (comma-separated)
    df_part = pd.read_csv(path)  # default sep="," and header=0
    print(f"{path} shape:", df_part.shape)
    print(f"{path} first columns:", df_part.columns[:10].tolist())
    dfs.append(df_part)

goe = pd.concat(dfs, ignore_index=True)
print("Combined shape:", goe.shape)
goe.head()


emotion_data/data/full_dataset/goemotions_1.csv shape: (70000, 37)
emotion_data/data/full_dataset/goemotions_1.csv first columns: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration']
emotion_data/data/full_dataset/goemotions_2.csv shape: (70000, 37)
emotion_data/data/full_dataset/goemotions_2.csv first columns: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration']
emotion_data/data/full_dataset/goemotions_3.csv shape: (71225, 37)
emotion_data/data/full_dataset/goemotions_3.csv first columns: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration']
Combined shape: (211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [8]:
# Cell 5: Define weights for each GoEmotions label → sensationalism

# All GoEmotions columns except metadata:
emotion_cols = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]

# Sensationalism weights (0 = not sensational, higher = more sensational)
emotion_weights = {
    # strongly sensational
    "anger":        1.0,
    "fear":         0.9,
    "disgust":      0.9,
    "excitement":   0.9,
    "surprise":     0.8,
    "disapproval":  0.8,
    "grief":        0.7,

    # somewhat sensational
    "annoyance":       0.6,
    "nervousness":     0.6,
    "embarrassment":   0.5,
    "confusion":       0.5,
    "disappointment":  0.4,
    "sadness":         0.4,
    "remorse":         0.3,
    "desire":          0.3,

    # not sensational / baseline tone
    "neutral":      0.0,
    "admiration":   0.0,
    "amusement":    0.0,
    "approval":     0.0,
    "caring":       0.0,
    "curiosity":    0.0,  # neutral-ish
    "gratitude":    0.0,
    "joy":          0.0,
    "love":         0.0,
    "optimism":     0.0,
    "pride":        0.0,
    "realization":  0.0,  # you explicitly said this should NOT be sensational
    "relief":       0.0,
}

# Sanity: ensure every emotion has a weight
missing = [c for c in emotion_cols if c not in emotion_weights]
print("Missing in weights:", missing)


Missing in weights: []


In [9]:
# Cell 6: Compute raw weighted sensationalism score

# Ensure we only keep rows that have at least one emotion label
goe["label_sum"] = goe[emotion_cols].sum(axis=1)
goe = goe[goe["label_sum"] > 0].copy()
print("After removing unlabeled rows:", goe.shape)

# Weighted raw score
def compute_raw_score(row):
    s = 0.0
    for emo in emotion_cols:
        if row[emo] == 1:
            s += emotion_weights.get(emo, 0.0)
    return s

goe["raw_sens_score"] = goe.apply(compute_raw_score, axis=1)

goe["raw_sens_score"].describe()


After removing unlabeled rows: (207814, 38)


count    207814.000000
mean          0.274224
std           0.425465
min           0.000000
25%           0.000000
50%           0.000000
75%           0.500000
max           4.700000
Name: raw_sens_score, dtype: float64

In [10]:
# Cell 7: Map raw score → 3-level sensationalism label

# For rows with any positive score
nonzero = goe.loc[goe["raw_sens_score"] > 0, "raw_sens_score"]
print("Nonzero raw score describe:\n", nonzero.describe())

# Choose quantile-based thresholds for mild vs high sensationalism
q_low, q_high = nonzero.quantile([0.4, 0.8])
q_low, q_high


Nonzero raw score describe:
 count    74856.000000
mean         0.761295
std          0.362961
min          0.300000
25%          0.500000
50%          0.800000
75%          0.900000
max          4.700000
Name: raw_sens_score, dtype: float64


(0.6, 0.9)

In [11]:
# Cell 8: Apply thresholds to create labels 0,1,2

def sens_label_from_raw(s, q_low, q_high):
    if s <= 0:
        return 0         # not sensational
    elif s <= q_low:
        return 1         # mild
    elif s <= q_high:
        return 2         # medium-high
    else:
        return 2         # very high → also class 2

goe["sens_label"] = goe["raw_sens_score"].apply(lambda s: sens_label_from_raw(s, q_low, q_high))

goe["sens_label"].value_counts(normalize=True)


sens_label
0    0.639793
2    0.195959
1    0.164248
Name: proportion, dtype: float64

In [12]:
# Cell 9: Train/val split

texts = goe["text"].astype(str).tolist()
labels = goe["sens_label"].values

X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

len(X_train_texts), len(X_val_texts)


(166251, 41563)

In [16]:
# Cell 10: Embed train/val texts with Gemini (this may take a bit but should be fine)

X_train = embed_texts_with_gemini(X_train_texts, batch_size=16)
X_val   = embed_texts_with_gemini(X_val_texts,   batch_size=16)

X_train.shape, X_val.shape


[embed_texts_with_gemini] ServerError on batch 23360:23376, attempt 1/3 → retrying after 2.0s


((166251, 768), (41563, 768))

In [13]:
# Cell 11: Train multinomial logistic regression

clf_sens = LogisticRegression(
    max_iter=1000,
    multi_class="multinomial",
    class_weight="balanced"  # helps if distribution isn't perfect
)

clf_sens.fit(X_train, y_train)

y_val_pred = clf_sens.predict(X_val)
print(classification_report(y_val, y_val_pred, digits=3))


NameError: name 'X_train' is not defined

In [18]:
# Cell 12: Save model & thresholds for reuse

import joblib
os.makedirs("models", exist_ok=True)

joblib.dump(clf_sens, "models/sensationalism_gemini_goemotions.joblib")

meta = {
    "q_low": float(q_low),
    "q_high": float(q_high),
    "emotion_weights": emotion_weights,
    "classes": [0, 1, 2],
}
import json
with open("models/sensationalism_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved models/sensationalism_gemini_goemotions.joblib")


Saved models/sensationalism_gemini_goemotions.joblib


In [19]:
# Cell 4: Load trained sensationalism model + meta
import joblib
import json
clf_sens = joblib.load("models/sensationalism_gemini_goemotions.joblib")

with open("models/sensationalism_meta.json", "r") as f:
    meta = json.load(f)

print("Loaded model and meta.")
print("Meta:", meta)


Loaded model and meta.
Meta: {'q_low': 0.6, 'q_high': 0.9, 'emotion_weights': {'anger': 1.0, 'fear': 0.9, 'disgust': 0.9, 'excitement': 0.9, 'surprise': 0.8, 'disapproval': 0.8, 'grief': 0.7, 'annoyance': 0.6, 'nervousness': 0.6, 'embarrassment': 0.5, 'confusion': 0.5, 'disappointment': 0.4, 'sadness': 0.4, 'remorse': 0.3, 'desire': 0.3, 'neutral': 0.0, 'admiration': 0.0, 'amusement': 0.0, 'approval': 0.0, 'caring': 0.0, 'curiosity': 0.0, 'gratitude': 0.0, 'joy': 0.0, 'love': 0.0, 'optimism': 0.0, 'pride': 0.0, 'realization': 0.0, 'relief': 0.0}, 'classes': [0, 1, 2]}


In [20]:
# Cell 13: Helper to score arbitrary texts with the trained model

def sens_probs_for_texts(texts):
    """
    texts: list[str]
    returns: np.ndarray shape (n_samples, 3) – probs for classes 0,1,2
    """
    X = embed_texts_with_gemini(texts, batch_size=32)
    probs = clf_sens.predict_proba(X)
    return probs

def sens_rating_from_probs(probs_row):
    """
    probs_row: array-like length 3 (p0,p1,p2)
    returns: rating in [0,100]
    """
    probs_row = np.asarray(probs_row, dtype=float)
    expected_class = np.dot(probs_row, np.array([0.0, 1.0, 2.0]))
    rating = (expected_class / 2.0) * 100.0
    return float(rating)

def score_text_sensationalism(text):
    """
    Single text → (rating 0..100, probs[3])
    """
    probs = sens_probs_for_texts([text])[0]
    rating = sens_rating_from_probs(probs)
    return rating, probs


In [21]:
# Cell 14: Quick checks on some toy texts

samples = [
    "You WON'T BELIEVE what happened!!",
    "The government announced a minor policy update today.",
    "I am devastated and furious about what they did.",
]

for t in samples:
    r, p = score_text_sensationalism(t)
    print(f"Text: {t}\n  Rating: {r:.2f}  Probs: {p}\n")


Text: You WON'T BELIEVE what happened!!
  Rating: 86.06  Probs: [0.0978245  0.08316912 0.81900638]

Text: The government announced a minor policy update today.
  Rating: 23.00  Probs: [0.68833482 0.16325038 0.1484148 ]

Text: I am devastated and furious about what they did.
  Rating: 83.40  Probs: [0.02110104 0.28970422 0.68919475]



In [22]:
# Cell 15: Simple sentence splitter

import re

def split_sentences(text: str):
    text = (text or "").strip()
    if not text:
        return []
    # Split on . ! ? followed by whitespace
    sents = re.split(r'(?<=[.!?])\s+', text)
    return [s for s in sents if len(s.split()) >= 2]


In [23]:
# Cell 16: Article-level scoring with headline/body separation

def score_article_sensationalism(
    article: dict,
    weight_headline: float = 0.6
):
    """
    article dict expects:
      - 'title': str (optional but recommended)
      - 'content': str

    Returns dict with:
      - overall_rating (0..100)
      - headline_rating
      - body_avg_rating
      - body_sentence_ratings (list[float])
      - body_sentences (list[str])
      - headline (str)
    """
    title = (article.get("title") or "").strip()
    body  = (article.get("content") or "").strip()

    body_sents = split_sentences(body)

    # Decide headline
    if title:
        headline = title
    else:
        headline = body_sents[0] if body_sents else ""
        body_sents = body_sents[1:] if len(body_sents) > 1 else []

    # Score headline
    headline_rating = 0.0
    if headline:
        headline_rating, _ = score_text_sensationalism(headline)

    # Score body sentences
    body_sentence_ratings = []
    if body_sents:
        probs = sens_probs_for_texts(body_sents)
        for pr in probs:
            body_sentence_ratings.append(sens_rating_from_probs(pr))

    if body_sentence_ratings:
        body_avg_rating = float(np.mean(body_sentence_ratings))
    else:
        body_avg_rating = headline_rating

    overall_rating = float(
        weight_headline * headline_rating +
        (1.0 - weight_headline) * body_avg_rating
    )

    return {
        "overall_rating": round(overall_rating, 2),
        "headline_rating": round(headline_rating, 2),
        "body_avg_rating": round(body_avg_rating, 2),
        "body_sentence_ratings": [round(x, 2) for x in body_sentence_ratings],
        "body_sentences": body_sents,
        "headline": headline,
    }


In [24]:
# Cell 17: Test article scoring

article = {
    "title": "Trump 'retired' a database tracking the most expensive weather disasters. Now it's back — and finding over $100B in losses",
    "source": "CNN",
    "author": "Andrew Freedman",
    "publication_date": "2025-10-22 10:00:00",
    "content": "The Billion-Dollar Weather and Climate Disasters Database, which the Trump administration\"retired\" in May, has relaunched outside of the government using the same methodology. In its first update at the new site, the database shows that the first six months of 2025 have been the most expensive first six months of any year since 1980. The Billion-Dollar Database tracks the financial costs of property and other infrastructure destroyed by extreme weather disasters in the United States, focusing on events that caused $1 billion or more in damages. So far, 2025 has racked up $101.4 billion in such losses. The climate research nonprofit Climate Centralnow hosts the databaseand makes this information available to insurers, policy makers, broadcast meteorologists and ordinary citizens. The database was rebuilt and will be maintained by its previous administrator Adam Smith, a former economist at the National Oceanic and Atmospheric Administration, the agency which used to host it. Smith found 14 billion-dollar disasters in the first half of this year, including the Los Angeles wildfires in January and a tornado outbreak across the central US in mid-March.More billion-dollar disasters are likely to be added to the list before 2025 is over. Without the database, the public would have no easy way to track the cost of extreme weather events, many of which are becoming more common and severe because of climate change. But climate change is not the sole reason the database shows an upward trend in both the number of billion-dollar disasters and the amount they cost. Population growth and an increase in the number of buildings in harm's way are the dominant factors, according to Smith. \"Either way you look at it, the rise in damages relates to human activities and choices, and so you need to use information in context to better evaluate future choices,\" he said. The frequency of billion-dollar disasters has particularly increased in the last decade, Smith said, occurring nearly twice as often compared to the 30-year inflation-adjusted average. Between 1980 and 2024, there were nine such disasters on average each year. In the past five years, that annual average has jumped to 24. Therecord for a single yearwas 28 events in 2023. In the first six months of 2025, the list of billion-dollar disasters is mostly comprised of severe thunderstorms and tornado outbreaks, reflecting the conspicuous absence of landfalling hurricanes so far this season. However, the LA fires in January cost $61.2 billion, making them the costliest wildfires in US history, according to Climate Central. Climate Central hired Smith after the NOAA economist took early retirement this year, as part of the Trump administration's push to shrink the federal bureaucracy. NOAA's official list stopsat the end of2024. Climate Central's version picks up in 2025 and will continue from there. The relaunched list uses the same methodology as the old NOAA one, Smith said. It relies on data from insurance companies and other sources, some of which is proprietary, to tally up total losses. The decision to discontinue the database was due in part to Smith's exit from NOAA. It would have been a difficult task for the agency to continue without him, but it could have done it, he said. The choice to discontinue the database was in keeping with the administration's focus on cutting climate change datasets and programs across federal agencies. But there were calls for it to continue from multiple sectors. \"This dataset was simply too important to stop being updated. Demand for its revival actually came from several aspects of industry and society, including decision makers in the insurance, reinsurance risk space, academia, Congress and local communities,\" Smith said."
}

res = score_article_sensationalism(article, weight_headline=0.6)
res


{'overall_rating': 41.46,
 'headline_rating': 45.4,
 'body_avg_rating': 35.56,
 'body_sentence_ratings': [40.75,
  58.34,
  26.79,
  41.07,
  21.51,
  27.92,
  44.41,
  32.36,
  38.43,
  25.51,
  42.45,
  38.99,
  38.86,
  45.23,
  38.25,
  46.51,
  44.59,
  27.75,
  33.0,
  21.52,
  22.74,
  31.89,
  42.32,
  41.88,
  34.4,
  25.1,
  36.01,
  27.12],
 'body_sentences': ['The Billion-Dollar Weather and Climate Disasters Database, which the Trump administration"retired" in May, has relaunched outside of the government using the same methodology.',
  'In its first update at the new site, the database shows that the first six months of 2025 have been the most expensive first six months of any year since 1980.',
  'The Billion-Dollar Database tracks the financial costs of property and other infrastructure destroyed by extreme weather disasters in the United States, focusing on events that caused $1 billion or more in damages.',
  'So far, 2025 has racked up $101.4 billion in such losses.',

In [25]:
article = {
    "title":  "Merkley nearly breaks Booker's filibuster record, wins his praise for fighting 'Trump's authoritarian tactics'",
    "source": "FOX",
    "author": "Leo Briceno",
    "publication_date": "2025-10-22T00:00:00",
    "content": "Democrats pulled out all the stops Wednesday to delay the vote on a short-term spending bill to reopen the government, the 12th time the Senate has considered the measure since the government entered a shutdown Oct. 1.\nSen. Jeff Merkley, D-Ore., embarked on a nearly 24-hour speech at 6:23 p.m. Tuesday, concluding his remarks at 5 p.m. the next day. Merkley, 68, warned viewers of the authoritarianism he said had become a facet of the Trump administration.\n\"Be aware and worried about the possibility of the use of an emergency in order to expand authoritarian power. That’s the position we’re in now in the United States of America. Authoritarianism with a rubber-stamp Congress, a court that’s delivering more and more power to the executive and an executive who has a well-planned strategy,\" Merkley said in his remarks.\nJOHNSON WARNS US 'BARRELING TOWARD ONE OF THE LONGEST SHUTDOWNS' IN HISTORY\n\"Republicans have shut down the government to continue the strategy of slashing Americans' healthcare.\"\nHe delivered his speech as lawmakers remain gridlocked over federal funding for 2026. While Republicans in the House of Representatives have passed a short-term funding bill to keep the government open through Nov. 21, Democrats in the Senate have voted a dozen times to defeat the package.\nThe Senate once again failed to advance the package on Wednesday. It failed in a 54-46 vote.\nDemocrats, led by Senate Majority Leader Chuck Schumer, D-N.Y., and House Minority Leader Hakeem Jeffries, D-N.Y., have demanded an extension of COVID-era supplemental funding for Obamacare healthcare subsidies that will sunset in 2025.\nSCREAMING MATCH ERUPTS BETWEEN HAKEEM JEFFRIES, MIKE LAWLER AS GOVERNMENT SHUTDOWN CHAOS CONTINUES\nRepublicans need the support of seven Democrats to overcome the 60-vote threshold to overcome a filibuster. The GOP holds 53 seats in the chamber.\nMerkley, who came close to breaking Sen. Cory Booker’s 25-hour and 4-minute record set earlier this year, put the shutdown blame on Republicans throughout his discourse.\nBooker praised Merkley's stalling efforts online.\n\"Listening to Senator Jeff Merkley for over 22 hours, it is clear that we need to stand up for our democracy. We must continue to call out and counter Trump's authoritarian tactics. Thank you, Jeff!\" Booker said in a post on X.\nOn the issue of authoritarianism, the primary topic of Merkley's remarks, Merkley decried what he saw as the Trump administration’s attempts to push the limits on executive power, like its deployment of the National Guard to urban areas.\n\"If you remove a clear standard as to whether there is a rebellion and just say a president can deploy the military on a whim in places he doesn’t like against peaceful protesters to distract Americans or to exercise a suppression of dissent, then you have flung the doors open to tyranny, to a strongman state,\" Merkley said.\nCLICK HERE TO GET THE FOX NEWS APP\nPresident Donald Trump has deployed the National Guard to Washington, D.C.; Los Angeles; Chicago; Memphis; and Portland, Oregon, citing a need to protect law enforcement and government operations in those cities.", "url": "https://www.foxnews.com/politics/merkley-nearly-breaks-bookers-filibuster-record-wins-his-praise-fighting-trumps-authoritarian-tactics", "label": "Republican"}


res = score_article_sensationalism(article, weight_headline=0.6)
res


{'overall_rating': 41.66,
 'headline_rating': 44.14,
 'body_avg_rating': 37.93,
 'body_sentence_ratings': [41.98,
  40.56,
  27.77,
  19.65,
  49.4,
  55.96,
  28.14,
  43.6,
  47.14,
  25.3,
  42.61,
  55.15,
  55.56,
  31.14,
  43.62,
  18.11,
  35.45,
  35.28,
  31.46,
  39.92,
  30.5,
  17.97,
  57.17,
  51.36,
  23.47],
 'body_sentences': ['Democrats pulled out all the stops Wednesday to delay the vote on a short-term spending bill to reopen the government, the 12th time the Senate has considered the measure since the government entered a shutdown Oct.',
  'Jeff Merkley, D-Ore., embarked on a nearly 24-hour speech at 6:23 p.m.',
  'Tuesday, concluding his remarks at 5 p.m.',
  'the next day.',
  'Merkley, 68, warned viewers of the authoritarianism he said had become a facet of the Trump administration.',
  '"Be aware and worried about the possibility of the use of an emergency in order to expand authoritarian power.',
  'That’s the position we’re in now in the United States of Ame

In [26]:
article = {
    "title":  "Sparks fly as Cuomo, Mamdani tear into each other during fiery debate: 'Toxic energy'",
    "source": "FOX",
    "author": "Alec Schemmel",
    "publication_date": "2025-10-22T00:00:00",
    "content": "Front-runners for New York City mayor, Zohran Mamdani and Andrew Cuomo, wasted little time attacking each other on alleged personal scandals they have been involved in during a Wednesday night debate between the pair and GOP candidate Curtis Sliwa.\nMamdani and Sliwa took the opportunity during Wednesday's debate to drill down on past sexual harassment allegations against Cuomo, the former governor of New York, ahead of an impeachment inquiry that preceded Cuomo's 2021 resignation. Cuomo was also hit by Mamdani over accusations he has – while in public office – failed to meet with Muslim constituents and only began doing so amid pressure from his mayoral campaign, and over his alleged poor handling of the COVID-19 virus in New York after Cuomo was party to issuing guidance forcing nursing homes and long-term care facilities to admit COVID-19 positive patients.\nMeanwhile, Cuomo did not hold back on targeting Mamdani over alleged controversies that have embattled his campaign. Cuomo blasted the self-proclaimed socialist over his lack of experience, ties to radical politics, and past radical comments about law enforcement, Israel and the situation in Gaza.\nFBI AGENTS FROM '93 WTC ATTACK BLAST MAMDANI FOR EMBRACING RADICAL IMAM\n\"My main opponent has no new ideas. He has no new plan. … He's never run anything, managed anything. He's never had a real job,\" Cuomo said of Mamdani during the debate. Cuomo also branded Mamdani as someone who has proven to be \"a divisive force in New York,\" pointing to past incidents that have garnered Mamdani heat from critics.\nOne of those incidents included a picture he took with a hard-lined Ugandan lawmaker who has pushed policies of imprisoning people for being gay, which Mamdani took while taking a break from the campaign trail to visit his home country of Uganda for a wedding. Cuomo also hit the controversy over whether Mamdani supports Jewish New Yorkers, as his critics have claimed he is anti-Israel pointing to statements he has made, like \"globalize the intifada.\"\nCuomo also accused Mamdani of disrespecting Italian-Americans after a video of him surfaced giving the middle finger to a statue of Christopher Columbus, while also pointing to criticism the self-proclaimed socialist candidate has garnered from 9/11 first-responders after posting a photo with a Muslim cleric who served as a character witness for the mastermind behind the September 11, 2001 attacks.\nTOP 5 MOMENTS FROM FIERY NYC MAYORAL DEBATE: 'HE LITERALLY HAS NEVER HAD A JOB'\n\"You have been a divisive force in New York, and I believe that's toxic energy for New York. It's with the Jewish community. It's with the Italian-American community – when you give the Columbus statue the finger. It's with the Sunni Muslims when you say decriminalize prostitution, which is Haram. It's the Hindus,\" Cuomo continued. \"Then, you take a picture with Rebecca Kadaga, deputy Prime Minister of Uganda. … She's known as Rebecca ‘Gay Killer.’ … You're a citizen of Uganda. You took the picture. You said you didn't know who she was. It turns out you did. How do you not renounce your citizenship or demand BDS against Uganda for imprisoning people who are gay just by their sexual orientation? Isn't that a basic violation of human rights?\"\nMamdani shot back that his politics have remained \"consistent\" and that they are built on a belief in human rights for all people, including LGBTQ+ folks. Had he known Kadga's role in drafting legislation to imprison gay folks, Mamdani said, he never would have taken the picture.\n\"This constant attempt to smear and slander me is an attempt to also distract from the fact that, unlike myself, you do not actually have a platform or a set of policies,\" Mamdani shot back at Cuomo before introducing his own claims about the former governor regarding past accusations of sexual harassment.\nMAMDANI RIPPED BY RIVALS FOR UNPOPULAR STANCE DURING FIERY NYC DEBATE: 'YOU WON'T SUPPORT ISRAEL'\n\"Mr. Cuomo. In 2021, 13 different women who worked in your administration credibly accused you of sexual harassment. Since then, you have spent more than $20 million in taxpayer funds to defend yourself, all while describing these allegations as entirely political,\" Mamdani said while attacking Cuomo Wednesday night.\n\"You have even gone so far as to legally go after these women. One of those women, Charlotte Bennett, is here in the audience this evening. You sought to access her private gynecological records. She cannot speak up for herself because you lodged a defamation case against her. I, however, can speak. What do you say to the 13 women that you sexually harassed?\"\nCuomo, in 2021, was accused of multiple incidents of sexual harassment that preceded his resignation as governor that year. A subsequent report from New York Attorney General Letitia James confirmed Cuomo \"sexually harassed multiple women from 2013 through 2020,\" while in January 2024, the U.S. Department of Justice announced it had reached a nearly $500,000 settlement with Cuomo's executive office over one of the claims. However, no criminal charges were ever filed against Cuomo, with some district attorneys citing insufficient evidence.\nCLICK HERE TO GET THE FOX NEWS APP\nCuomo defended himself against Mamdani's accusations, noting the cases were eventually dropped, before returning to questions about Mamdani's alleged past.\nMeanwhile, Sliwa didn't skip an opportunity to slam Cuomo over the sexual assault allegations either, saying early in the debate during a discussion about homelessness that Cuomo \"fled\" the governor's office amid an impeachment inquiry that was investigating him.\n\"Andrew, you didn't ‘leave.’ You fled from being impeached by the Democrats in the state legislature,\" Sliwa began before getting into the homelessness issue, earning him a round-of-applause from the audience.\n\"'Leave?' You fled!\" Sliwa continued to applause. \"But let's get back on topic.\"", "url": "https://www.foxnews.com/politics/sparks-fly-cuomo-mamdani-tear-into-each-other-during-fiery-debate", "label": "Republican"}


res = score_article_sensationalism(article, weight_headline=0.6)
res


{'overall_rating': 45.71,
 'headline_rating': 49.23,
 'body_avg_rating': 40.43,
 'body_sentence_ratings': [38.39,
  32.62,
  39.12,
  34.03,
  58.14,
  44.97,
  54.76,
  44.59,
  30.83,
  43.22,
  41.32,
  32.46,
  64.1,
  29.46,
  47.38,
  37.46,
  42.57,
  23.71,
  53.41,
  14.88,
  45.46,
  12.5,
  66.38,
  43.64,
  35.57,
  44.62,
  55.09,
  46.83,
  32.25,
  39.04,
  59.09,
  46.09,
  45.78,
  32.87,
  41.53,
  44.78,
  22.5,
  30.37,
  29.64,
  31.78,
  46.48,
  43.08,
  35.71],
 'body_sentences': ['Front-runners for New York City mayor, Zohran Mamdani and Andrew Cuomo, wasted little time attacking each other on alleged personal scandals they have been involved in during a Wednesday night debate between the pair and GOP candidate Curtis Sliwa.',
  "Mamdani and Sliwa took the opportunity during Wednesday's debate to drill down on past sexual harassment allegations against Cuomo, the former governor of New York, ahead of an impeachment inquiry that preceded Cuomo's 2021 resignation

In [27]:
article = {
    "title":  "The latest on Donald Trump’s many legal clouds",
    "source": "CNN",
    "author": "Dan Berman, Zachary B. Wolf",
    "publication_date": "2023-10-02T00:00:00",
    "content": "Former President Donald Trump has been campaigning in between his many different court appearances for much of the year.\nBut his decision to attend the first day of his $250 million civil fraud trial in New York created another opportunity to appear on camera from inside a courtroom when the judge allowed photographers to document the moment before proceedings got underway.\nKeeping track of the dizzying array of civil and criminal cases is a full-time job.\nHe is charged with crimes related to conduct:\n- Before his presidency – a hush money scheme that may have helped him win the White House in 2016.\n- During his presidency – his effort to stay in the White House by overturning the 2020 election.\n- After his presidency – his treatment of classified material and alleged attempts to hide it from the National Archives.\nTrump denies any wrongdoing and has pleaded not guilty in all of the criminal cases. He alleges a “witch hunt” against him. But each trial has its own distinct storyline to follow.\nHere’s an updated list of developments in Trump’s very complicated set of court cases, beginning with the one playing out in Manhattan this week.\nNew York state civil court: $250 million fraud case\nThe civil fraud trial, unlike Trump’s multiple criminal indictments, does not carry the danger of a felony conviction and jail time, but it could very well cost him some of his most prized possessions, including Trump Tower.\nNew York Attorney General Letitia James brought the $250 million lawsuit in September 2022, alleging that Trump and his co-defendants committed repeated fraud in inflating assets on financial statements to get better terms on commercial real estate loans and insurance policies.\nJudge Arthur Engoron has already ruled that Trump and his adult sons are liable for fraud for inflating the value of his golf courses, hotels and homes on financial statements to secure loans.\nThe trial portion of the case, playing out in court in Manhattan, will assess what damages will be levied against Trump and how Engoron’s decision to strip Trump of his New York business licenses will play out.\nFederal civil court in New York: E. Jean Carroll defamation lawsuit\nIn May, a federal jury in Manhattan found Trump sexually abused former advice columnist E. Jean Carroll in a luxury department store dressing room in the mid-1990s and awarded her about $5 million.\nA separate civil defamation lawsuit will only need to decide how much money Trump has to pay her. That case for January 15 – the same day Iowa Republicans will hold their caucuses, the first date on the presidential primary calendar.\nFederal criminal court in DC: 2020 election Interference\nIn August, Trump was indicted by a federal grand jury in special counsel Jack Smith’s investigation into the aftermath of the 2020 election. The former president was arraigned in a Washington, DC, courtroom, where he pleaded not guilty.\nThe case is based in part on a scheme to create slates of fake electors in key states won by President Joe Biden.\nIn late September, Judge Tanya Chutkan rejected Trump’s request that she recuse herself from the case. Chutkan, a Barack Obama appointee, has overseen civil and criminal cases related to the January 6, 2021, insurrection and has repeatedly exceeded what prosecutors have requested for convicted rioters’ prison sentences.\nChutkan set the trial’s start date for March 4, 2024, the day before Super Tuesday, when the largest batch of presidential primaries will occur. The trial marks the first of Trump’s criminal cases expected to proceed.\nNew York criminal court: Hush money payments\nTrump has been charged in Manhattan criminal court with 34 felony counts of falsifying business records related to his role in a hush money payment scheme involving adult film actress Stormy Daniels late in the 2016 presidential campaign.\nThe former president pleaded not guilty at his April arraignment in Manhattan.\nProsecutors, led by Manhattan District Attorney Alvin Bragg, accuse Trump of falsifying business records with the intent to conceal $130,000 in payments to Daniels made by former Trump attorney and fixer Michael Cohen to guarantee her silence about an alleged affair.\nTrump has denied having an affair with Daniels.\nThe trial was originally scheduled to begin in late March 2024, but Judge Juan Merchan has suggested the date could move. The next court date is scheduled for February.\nGeorgia criminal court: Efforts to overturn election results\nFulton County District Attorney Fani Willis is using racketeering violations to charge a broad criminal conspiracy against Trump and 18 others in their efforts to overturn Biden’s victory in Georgia.\nThe probe was launched in 2021 following Trump’s call that January with Georgia Secretary of State Brad Raffensperger, in which the president pushed the Republican official to “find” votes to overturn the election results.\nThe August indictment also includes how Trump’s team allegedly misled state officials in Georgia; organized fake electors; harassed an election worker; and breached election equipment in rural Coffee County, Georgia.\nOne co-defendant, bail bondsman Scott Hall, has pleaded guilty to five counts in the case.\nFulton County prosecutors have signaled they could offer plea deals to other co-defendants.\nWillis this week issued a subpoena to former New York City Police Commissioner Bernard Kerik, a Trump ally, who in turn demanded an immunity deal in exchange for testimony.\nTrial for two co-defendants is expected to begin this month and could last three to five months. A trial date has not been set for Trump, who has pleaded not guilty.\nFederal criminal court in Florida: Mishandling classified material\nTrump has pleaded not guilty to 37 federal charges brought by Smith over his alleged mishandling of classified documents. Smith added three additional counts in a superseding indictment.\nThe investigation centers on sensitive documents that Trump brought to his Mar-a-Lago residence in Florida after his White House term ended in January 2021.\nThe National Archives, charged with collecting and sorting presidential material, has previously said that at least 15 boxes of White House records were recovered from Mar-a-Lago, including some classified records.\nTrump was also caught on tape in a 2021 meeting in Bedminster, New Jersey, where the former president discussed holding secret documents he did not declassify.\nSmith’s additional charges allege that Trump and his employees attempted to delete Mar-a-Lago security footage sought by the grand jury investigating the mishandling of the records.\nTrial is not expected until May, after most presidential primaries have concluded.\nThere are other cases to note:\nTrump Organization: Convicted of criminal tax fraud\nTrump’s namesake business, the Trump Organization, was convicted in December by a New York jury of tax fraud, grand larceny and falsifying business records in what prosecutors say was a 15-year scheme to defraud tax authorities by failing to report and pay taxes on compensation provided to employees.\nManhattan prosecutors told a jury the case was about “greed and cheating,” laying out a scheme within the Trump Organization to pay high-level executives in perks such as luxury cars and apartments without paying taxes on them.\nFormer Trump Organization Chief Financial Officer Allen Weisselberg pleaded guilty to his role in the tax scheme. He was released after serving four months in jail at Rikers Island.\nJanuary 6: Lawsuits by police officers\nSeveral members of the US Capitol Police and Washington, DC, Metropolitan Police are suing Trump, saying his words and actions incited the 2021 riot.\nThe various cases accuse Trump of directing assault and battery; aiding and abetting assault and battery; and violating Washington laws that prohibit the incitement of riots and disorderly conduct.\nIn August, Trump requested to put on hold the lawsuit related to the death of Capitol Police Officer Brian Sicknick, citing his various criminal trials. The estate of Sicknick, who died after responding to the attack on the Capitol, is suing two rioters involved in the attack and Trump for his alleged role in egging it on.\nOther lawsuits have been put on hold while a federal appeals court considers whether Trump had absolute immunity as the sitting president.\nPersonal retaliation: Peter Strzok lawsuit\nFormer top FBI counterintelligence official Peter Strzok, who was fired in 2018 after the revelation that he criticized Trump in text messages, sued the Justice Department, alleging he was terminated improperly.\nIn summer 2017, former special counsel Robert Mueller removed Strzok from his team investigating Russian interference in the 2016 election after an internal investigation revealed texts with former FBI lawyer Lisa Page that could be read as exhibiting political bias.\nStrzok and Page were constant targets of verbal attacks by Trump and his allies, part of the larger ire the then-president expressed toward the FBI during the Russia investigation. Trump repeatedly and publicly called for Strzok’s ouster until he was fired in August 2018.\nTrump is set to be deposed this month as part of the case, according to Politico.\nTrump lawsuit against Hillary Clinton, Democrats, ex-FBI officials is dismissed\nA federal judge dismissed Trump’s lawsuit against Hillary Clinton, the Democratic National Committee, several ex-FBI officials and more than two dozen other people and entities that he claims conspired to undermine his 2016 campaign with fabricated information tying him to Russia.\n“What (Trump’s lawsuit) lacks in substance and legal support it seeks to substitute with length, hyperbole, and the settling of scores and grievances,” US District Judge Donald Middlebrooks wrote.\nTrump appealed the decision, but Middlebrooks also ruled that the former president and his attorneys are liable for nearly $1 million in sanctions for bringing the case.\nTrump launched a Hail Mary bid in July to revive the sprawling lawsuit, relying on a recent report from special counsel John Durham that criticized the FBI’s Trump-Russia probe.\nTrump victory in Michael Cohen’s retaliation suit\nTrump’s former lawyer Cohen sued Trump, former Attorney General William Barr and others, alleging they put him back in jail to prevent him from promoting his upcoming book while under home confinement.\nCohen was serving the remainder of his sentence for lying to Congress and campaign violations at home, due to Covid-19 concerns, when he started an anti-Trump social media campaign in summer 2020. Cohen said that he was sent back to prison in retaliation and that he spent 16 days in solitary confinement.\nA federal judge threw out the lawsuit in November. District Judge Lewis Liman said he was empathetic to Cohen’s position but that Supreme Court precedent bars him from allowing the case to move forward.\nTrump-filed lawsuits: Bob Woodward\nTrump sued journalist Bob Woodward in January for alleged copyright violations, claiming Woodward released audio from their interviews without Trump’s consent.\nWoodward and publisher Simon & Schuster said Trump’s case is without merit and moved for its dismissal.\nWoodward conducted several interviews with Trump for his book “Rage,” published in September 2020. Woodward later released “The Trump Tapes,” an audiobook featuring eight hours of raw interviews with Trump interspersed with the author’s commentary.\nTrump-filed lawsuits: The New York Times, Mary Trump and CNN\nA federal judge in Florida dismissed Trump’s $475 million lawsuit against CNN that accused the network of defaming him by using the phrase “Big Lie” and allegedly comparing him to Adolf Hitler.\nA New York judge dismissed The New York Times from Trump’s lawsuit regarding disclosure of his tax returns and ordered Trump to pay the newspaper’s legal fees. Trump is still suing his niece Mary Trump for disclosure of the tax documents. She had tried to sue him for defrauding her out of millions after the death of his father, but the suit was dismissed.\nCNN’s Kara Scannell, Tierney Sneed, Katelyn Polantz and Marshall Cohen contributed to this report.", "url": "https://www.cnn.com/2022/11/15/politics/donald-trump-investigations-lawsuits/index.html", "label": "Democrat"}


res = score_article_sensationalism(article, weight_headline=0.6)
res


{'overall_rating': 28.03,
 'headline_rating': 25.39,
 'body_avg_rating': 32.0,
 'body_sentence_ratings': [21.27,
  28.31,
  39.38,
  22.01,
  11.08,
  25.91,
  17.61,
  51.51,
  28.59,
  33.21,
  46.85,
  37.58,
  25.62,
  25.98,
  14.87,
  28.02,
  24.98,
  24.32,
  26.39,
  35.49,
  27.31,
  13.07,
  27.0,
  40.25,
  31.16,
  20.21,
  50.74,
  26.25,
  36.75,
  29.97,
  37.03,
  35.95,
  32.68,
  24.94,
  21.84,
  37.75,
  23.27,
  25.19,
  38.42,
  16.67,
  46.27,
  34.52,
  27.6,
  33.01,
  29.06,
  38.33,
  26.82,
  45.16,
  42.32,
  40.05,
  23.68,
  43.93,
  37.56,
  33.04,
  49.82,
  26.98,
  56.81,
  42.84,
  55.35,
  33.72,
  24.29,
  40.08,
  26.61,
  35.76,
  40.53,
  20.25,
  20.15,
  35.9,
  31.21,
  38.41,
  25.22,
  29.32,
  32.46,
  20.06,
  40.63,
  36.75,
  36.26,
  44.95,
  16.95],
 'body_sentences': ['Former President Donald Trump has been campaigning in between his many different court appearances for much of the year.',
  'But his decision to attend the first day